# Using the Config Client

This is a simple tutorial on how to use the daq config server client inside dodal. 

The config server should already be installed in your venv. Import the client.

In [51]:
from daq_config_server.client import ConfigClient

The config server is deployed centrally in Argus. The end point for this is https://daq-config.diamond.ac.uk, but you could also deploy a version the config server elsewhere, such as on a beamline cluster.

Initialise an instance of the client with the appropriate endpoint.

In [52]:
CONFIG_SERVER_ENDPOINT = "https://daq-config.diamond.ac.uk"

config_client = ConfigClient.from_url(url=CONFIG_SERVER_ENDPOINT)

For some files, converters exist in the server that convert the contents of the file to a standard `dict` format or `Pydantic` model. For example, a model has been created for XPDF crystals on i15-1.

In [53]:
from daq_config_server.models.i15_1 import XpdfCrystalLookupTable

Now we can use the `get_file_contents` method to retrieve a config file's contents. Here you should provide the filepath of the file to read, and your desired return type.

In [54]:
crystal_config = config_client.get_file_contents(
    "/dls_sw/i15-1/software/daq_configuration/xpdf_crystal_lut.txt",
    XpdfCrystalLookupTable,
)

And now we have an instance of `XpdfCrystalLookupTable`, and can use that model's methods and attributes to get the information inside our config file.

In [55]:
crystal_config.get_column_names()

['y_mm', 'energy_keV']

In [56]:
crystal_config.columns

[[1.455, -48.845, 50.6], [40.05, 65.4, 76.69]]

In [57]:
crystal_config.rows

[[1.455, 40.05], [-48.845, 65.4], [50.6, 76.69]]

In [58]:
crystal_config.get_value("y_mm", 1.455, "energy_keV")

40.05

Below are examples of the errors you would get if the config file doesn't exist, isn't in the whitelist, or the wrong return type is requested.

In [59]:
config_client.get_file_contents(
    "/fake_path", str
)

/fake_path is not a whitelisted file.


HTTPError: /fake_path is not a whitelisted file.

In [ ]:
config_client.get_file_contents(
    "/dls_sw/i15-1/software/daq_configuration/fake_file", str
)


File /dls_sw/i15-1/software/daq_configuration/fake_file cannot be found


HTTPError: File /dls_sw/i15-1/software/daq_configuration/fake_file cannot be found

In [ ]:
from daq_config_server.models.lookup_tables import GenericLookupTable

detector_config = config_client.get_file_contents(
    "/dls_sw/i15-1/software/daq_configuration/xpdf_crystal_lut.txt",
    GenericLookupTable,
)

ValidationError: 1 validation error for GenericLookupTable
column_names
  Field required [type=missing, input_value={'rows': [[1.455, 40.05],..., 65.4], [50.6, 76.69]]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing

For more information on the config server see https://diamondlightsource.github.io/daq-config-server/main/index.html